# Task 15b — Alternative-only concept supervision and contrastive calibration

This notebook starts **only after Task 15 passes**. It adds two objectives to the same Zhang C4 pipeline and scores them against every Task 15 arm.

Terminology used throughout (the word "gold" is not used):

- **observed target** — the token that appeared in the C4 sentence (`chaos` in *"The room descended into chaos."*)
- **generated alternatives** — the extractor's substitute candidates (`turmoil, mayhem, disorder`)
- **human-accepted alternatives** — SWORDS substitutes accepted by annotators
- **target-inclusive set** — `{chaos, turmoil, mayhem, disorder}`; what Zhang's set-marginal supervises
- **alternative-only set** — `{turmoil, mayhem, disorder}`; what our concept term supervises. The observed target is still trained, by ordinary NTP at the slot (`--slot-ntp-weight 1.0`).

| Method | Question |
|---|---|
| Alternative-only uniform | Raise every generated alternative equally, with the observed target trained only by NTP: does probability transfer to *human-accepted* alternatives without the entropy cost the randomized control pays? |
| Target-inclusive uniform (ablation) | Same loss with the observed target inside the concept set: does excluding it matter? |
| Alternative-only + contrastive | Add InfoNCE over alternatives ∪ conservative same-POS negatives: does the model learn which plausible candidates should *not* receive that probability (SWORDS AUROC)? |

$$L = L_{\mathrm{NTP}} + \alpha\Big(-\tfrac{1}{|A(x)|}\sum_{a\in A(x)}\log p(a\mid x)\Big) + \beta\Big[\log\!\!\sum_{c\in A(x)\cup N(x)}\!\!e^{s(c)} - \log\!\sum_{a\in A(x)}e^{s(a)}\Big]$$

The contrastive term is joint token-level training; it is not DPO and not sentence-level SimCSE. Its positive set is the same filtered candidate set the concept term uses, so `--exclude-target` makes both alternative-only at once.

Why this and not more set-marginal: at 3B, Zhang's set-marginal matches a low-weight *randomized* control on SWORDS GAP and AUROC (paired CI crosses zero) and does not lower NLL on human-accepted alternatives at all (−0.05, n.s.), while the randomized arm lowers it by a full nat by flattening everything. The metrics that separate them are STS and perplexity, not SWORDS ranking. The method here has to move human-accepted alternatives without that entropy cost, and has to beat the randomized control, not merely NTP.

Every table carries Task 15's arms — pretrained, NTP, augmented NTP, randomized at $\lambda=.25$ and at the matched $\lambda=1$, and set-marginal at $\lambda=1$ — read from that notebook's manifest.


In [ ]:
from pathlib import Path
from getpass import getpass
import datetime, hashlib, json, os, re, shutil, subprocess, sys, torch

BASE_MODEL = "meta-llama/Llama-3.2-1B"
# Every per-model artifact -- data, adapters, manifests, results -- is keyed on
# this tag.  Nothing about a model may share a path with another model: adapters
# are restored from Drive by path, so a collision would silently hand one model's
# weights to another and the run would look like it succeeded.
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
# combined.jsonl depends only on the tokenizer (plus spaCy), so a model that
# shares a vocabulary with one already extracted produces a byte-identical file.
# Llama-3.2 1B and 3B do.  Naming the donor here skips the POS pass, which is the
# most expensive stage of extraction; the vocabularies are compared before the
# copy and the run aborts if they differ.
REUSE_CONTENT_WORDS_FROM = None
PRIMARY_SEED = 42
# One seed first.  Seed 42 alone screens the pipeline and shows the direction of
# every effect, but it CANNOT support a claim: the pre-registered rule needs all
# three seeds to agree in sign.  Flip to True for the reportable run; resume makes
# the seed-42 arms free the second time.
RUN_MULTISEED = False
SEEDS = [PRIMARY_SEED] + ([123, 2024] if RUN_MULTISEED else [])
UPSTREAM_COMMIT = "b1d414143d11c8ed988b4cccbb06626cc8272bbe"
MAIN = Path("/content/concept-aware-training")
EXT = Path("/content/learning-concepts")
DATA = Path("/content/concept_data")
RUNS = Path("/content/concept_runs")
DRIVE_ROOT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_ROOT / "MyDrive/concept_training"
DRIVE_RESULTS = DRIVE_PROJECT / "task15_16_results"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"

# Turn on one gate at a time.  Defaults are safe and do not start a GPU matrix.
RUN_DATA = False
RUN_SMOKE = False
RUN_SCREEN = False
RUN_CONFIRM = False
# Task 15b only: Zhang's set marginal plus an alternative-only auxiliary term, and
# the two negative-quality controls.  Seed 42 unless SELECTED_HYBRID is set.
RUN_HYBRID = False
# The two negative-quality controls (clean / fragments).  They are a diagnostic
# for the demoted contrastive term, NOT a method-selection run, so they are off
# by default and should be run AFTER the H1/H2 screen has been gated.
RUN_NEGATIVE_CONTROLS = False
RUN_EVAL = False
# Skip any run whose adapter is already on Drive.  /content is wiped between
# Colab sessions, so without this the screen -> gate -> confirm sequence has to
# finish in one sitting.  Set False to force a full retrain.
RESUME_FINISHED_RUNS = True
# Precision for CONCEPT EXTRACTION only; training is QLoRA-4bit either way.
# Upstream inherits use_4bit=True from TrainingConfig, but extraction runs ~94
# small forwards per sequence and NF4 dequantization dominates them: 8.0 s/seq
# at 4-bit against roughly a third of that in bf16, i.e. 22 h against ~8 h for
# ten shards.  Quantization also perturbs the top-100 pool and the 0.75 cosine
# threshold the method depends on.  Use ONE setting for all ten shards.
EXTRACT_4BIT = False
# How many C4 sequences to extract concept sets for.  The paper uses 10,000 split
# 8000/1000/1000, but extraction measured 7.45 s/sequence on an L4 (spaCy 32%,
# GPU the rest; bf16 moved it only 7%), so the full set is ~21 h for one model.
# Zhang et al. Fig. 8 ablates exactly this and reports STS unchanged at a quarter
# of the training data, so 4,000 keeps the 80/10/10 ratio at ~8 h.  Set 10000 for
# the strict reproduction.  merge_synonym_parts hard-fails unless the split sizes
# sum to the number of sequences the shards actually cover, and the 3B scaling
# stage uses the same count so size comparisons are not confounded by data volume.
EXTRACT_SEQUENCES = 4000
# Sequences per extraction shard.  A shard is the unit of resume: it is cached to
# Drive only once it completes, so this is exactly what a disconnect costs.  At
# 3B's measured 10 s/sequence a 1000-shard is nearly three hours of exposure;
# 500 halves that for one extra model load per shard, about 30 seconds.
EXTRACT_SHARD = 500
SPLIT_TRAIN = int(EXTRACT_SEQUENCES * 0.8)
SPLIT_VAL = SPLIT_TEST = int(EXTRACT_SEQUENCES * 0.1)

_BAR = re.compile(r"\b(\d+)/(\d+)\s*\[")     # bounded tqdm: "  200/1000 ["
_BAR_OPEN = re.compile(r"\b(\d+)it\s*\[")     # unbounded tqdm: "  3200it [00:49"
PROGRESS_EVERY = 100                          # one line per this many items

def run(argv, cwd=None, env=None):
    """Run a child process, streaming its output into the cell.

    subprocess.run() writes the child's stdout to the kernel's file descriptor,
    which Colab does not route into the cell, so a failing command used to raise
    CalledProcessError with no diagnostic at all.  Stream it line by line and put
    the tail into the exception message.
    """
    argv = list(map(str, argv))
    print("+", " ".join(argv), flush=True)
    merged = os.environ.copy()
    merged.update({"CONCEPT_DATA_ROOT": str(DATA),
                   "CONCEPT_CHECKPOINT_ROOT": str(RUNS),
                   "CONCEPT_RESULTS_ROOT": str(DRIVE_RESULTS)})
    if env: merged.update(env)
    process = subprocess.Popen(argv, cwd=cwd, env=merged, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        line = line.replace("\r", "")
        tail.append(line)
        del tail[:-40]
        hit = _BAR.search(line)       # tqdm writes "c4:  12%| | 120/1000 [..."
        if hit:
            done, total = int(hit.group(1)), int(hit.group(2))
            if done % PROGRESS_EVERY and done != total:
                continue
        else:
            # Dataset loading has no total ("3200it [00:49"), so there is no final
            # count to anchor on.  Thin it ten times harder; it is pure noise and
            # seven training runs would otherwise emit tens of thousands of lines.
            loose = _BAR_OPEN.search(line)
            if loose and int(loose.group(1)) % (PROGRESS_EVERY * 10):
                continue
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(
            f"command failed with exit code {code}\n  {' '.join(argv)}\n"
            f"--- last {len(tail)} lines of its output ---\n{''.join(tail)}")

def assert_ephemeral(path):
    resolved = str(Path(path).resolve())
    assert resolved.startswith("/content/") and not resolved.startswith("/content/drive/"), resolved

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def sync_small_artifacts(source, label):
    destination = DRIVE_RESULTS / label
    destination.mkdir(parents=True, exist_ok=True)
    for path in Path(source).rglob("*"):
        if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png", ".log"}:
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

def audit_no_drive_weights():
    # LoRA adapters at r=4 are ~10 MB and ARE cached to Drive on purpose: /content is
    # ephemeral, and without them a disconnect between the screen and confirm phases
    # discards every trained arm.  Full weights and optimizer state stay off Drive.
    forbidden = {"pytorch_model.bin", "model.safetensors", "optimizer.pt",
                 "scheduler.pt", "scaler.pt", "rng_state.pth"}
    found = [str(p) for p in DRIVE_RESULTS.rglob("*")
             if p.name in forbidden or p.name.startswith("checkpoint-")]
    assert not found, f"full-weight/optimizer artifacts reached Drive: {found}"

DATA_CACHE = DRIVE_PROJECT / "task15_16_data"

def cache_dataset_to_drive(leaf):
    """Cache JSONL data needed by later notebooks, including top-k shards."""
    model_root = Path(leaf).parent
    copied = 0
    for source in (model_root, model_root / "embedding", model_root / "prompting"):
        destination = DATA_CACHE / source.relative_to(DATA)
        destination.mkdir(parents=True, exist_ok=True)
        for path in source.glob("*.jsonl"):
            shutil.copy2(path, destination / path.name)
            copied += 1
    print(f"cached {copied} dataset files under", DATA_CACHE / model_root.relative_to(DATA))

def restore_dataset_from_drive(leaf):
    model_root = Path(leaf).parent
    embedding_source = DATA_CACHE / Path(leaf).relative_to(DATA)
    copied = 0
    for destination in (model_root, model_root / "embedding", model_root / "prompting"):
        source = DATA_CACHE / destination.relative_to(DATA)
        if not source.exists():
            continue
        destination.mkdir(parents=True, exist_ok=True)
        for path in source.glob("*.jsonl"):
            target = destination / path.name
            if not target.is_file():
                shutil.copy2(path, target)
                copied += 1
    print(f"restored {copied} dataset files from", DATA_CACHE / model_root.relative_to(DATA))
    return (embedding_source / "synonyms_train.jsonl").is_file()

# The 1B reproduction wrote these names before the study covered more than one
# model; they stay as they are so that work still resolves, and every other model
# gets its own suffix rather than overwriting it.
_TAG_SUFFIX = "" if MODEL_TAG == "llama-3.2-1b" else f"_{MODEL_TAG}"
RESULT_DIR = f"task15_reproduction{_TAG_SUFFIX}"
LOG_DIR = f"task15_logs{_TAG_SUFFIX}"

# HyperLex is split train/dev/test; the lexical split shares no lemma between
# them, which is the stricter generalisation setting.  Every diagnostic in these
# notebooks reads DEV.  The test file is deliberately not referenced anywhere:
# the hierarchy claim in Task 16 is scored on it exactly once, after the method
# is locked, and a diagnostic that peeks at it would spend that.
HYPERLEX_DEV = MAIN / "data/hyperlex-data/splits/lexical/hyperlex_dev_all_lexical.txt"
HYPERLEX_TEST = MAIN / "data/hyperlex-data/splits/lexical/hyperlex_test_all_lexical.txt"  # locked
RUN_MANIFEST = DRIVE_RESULTS / "run_manifests"

def save_runs(runs, name):
    """Persist label -> adapter path so a later session can evaluate earlier phases."""
    RUN_MANIFEST.mkdir(parents=True, exist_ok=True)
    (RUN_MANIFEST / f"{name}.json").write_text(
        json.dumps({k: str(v) for k, v in runs.items()}, indent=2))

def load_runs(name):
    path = RUN_MANIFEST / f"{name}.json"
    if not path.is_file():
        return {}
    return {k: Path(v) for k, v in json.loads(path.read_text()).items()}

def restore_all(runs):
    """Pull every adapter in `runs` back onto /content; drop any that is missing."""
    live = {}
    for label, path in runs.items():
        if (Path(path) / "adapter_config.json").is_file() or restore_adapter_from_drive(path):
            live[label] = Path(path)
        else:
            print("missing adapter, dropping from this pass:", label)
    return live

def eval_done(marker):
    """True when a completed evaluation artifact is already on Drive."""
    return Path(marker).is_file() and RESUME_FINISHED_RUNS

def eval_covered(path, checkpoints):
    """True when `path` already scores every checkpoint of THIS pass.

    Each JSON evaluator writes a list of {"checkpoint": ..., ...}.  Testing
    coverage rather than mere existence is what makes this safe to resume:
    adding a seed grows `checkpoints`, the old file no longer covers it, and the
    evaluator reruns.  A plain "file exists" check would instead report the
    previous pass's table as if it were this one's.
    """
    if not (RESUME_FINISHED_RUNS and Path(path).is_file()):
        return False
    try:
        rows = json.loads(Path(path).read_text())
    except (json.JSONDecodeError, OSError):
        return False              # truncated by a disconnect mid-write; redo it
    if not isinstance(rows, list):
        return False
    scored = {str(row.get("checkpoint")) for row in rows if isinstance(row, dict)}
    return set(map(str, checkpoints)) <= scored

def sts_covered(path):
    """True when one STS pass already wrote its nine task rows to `path`."""
    if not (RESUME_FINISHED_RUNS and Path(path).is_file()):
        return False
    with open(path, encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip()) >= 10   # header + 9 tasks

def guarded(path, checkpoints, argv, cwd, what):
    """Run one evaluator unless its output already covers every checkpoint."""
    if eval_covered(path, checkpoints):
        print(f"resume: {what} already covers {len(checkpoints)} checkpoints, skipping")
        return
    run(argv, cwd=cwd)

ADAPTER_CACHE = DRIVE_PROJECT / "task15_16_adapters"
# train.py writes concept_training_config.json (the objective metadata: lambda,
# alpha, gamma, exclude_target, seed).  It was missing here, so a session that
# died after training cached the WEIGHTS but not the description of what they
# were trained with -- recoverable only from the directory name.  run_config.json
# is kept for older runs that wrote it.
ADAPTER_FILES = ("adapter_config.json", "adapter_model.safetensors",
                 "training_history.jsonl", "run_config.json",
                 "concept_training_config.json")

def adapter_cache_dirs(path):
    """Drive locations for one adapter, most current first.

    The second entry is the layout used before adapters were keyed on the model,
    so the seed-42 arms trained then still resume instead of silently retraining.
    """
    relative = Path(path).relative_to(RUNS)
    locations = [ADAPTER_CACHE / relative]
    # The pre-tag layout was written by the 1B reproduction and by nothing else,
    # so only that model may look there.  Offering it to every model would let 3B
    # restore 1B's weights and report them as a 3B result.
    if not _TAG_SUFFIX:
        locations.append(ADAPTER_CACHE / Path(*relative.parts[1:]))
    return locations

def cache_adapter_to_drive(path):
    """Copy one finished adapter to Drive so a later session can resume."""
    destination = adapter_cache_dirs(path)[0]
    destination.mkdir(parents=True, exist_ok=True)
    for name in ADAPTER_FILES:
        source = Path(path) / name
        if source.is_file():
            shutil.copy2(source, destination / name)
    return destination

def restore_adapter_from_drive(path):
    """Return True when a completed adapter for `path` was restored from Drive."""
    for source in adapter_cache_dirs(path):
        if not (source / "adapter_config.json").is_file():
            continue
        Path(path).mkdir(parents=True, exist_ok=True)
        for name in ADAPTER_FILES:
            candidate = source / name
            if candidate.is_file():
                shutil.copy2(candidate, Path(path) / name)
        return True
    return False

DATA.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
# DRIVE_RESULTS is deliberately NOT created here.  Creating any path under
# /content/drive before drive.mount() makes the mountpoint non-empty, and the
# mount then fails with "Mountpoint must not already contain files".  The next
# cell creates it immediately after mounting.


In [ ]:
from google.colab import drive
if DRIVE_ROOT.is_dir() and not (DRIVE_ROOT / "MyDrive").is_dir():
    # A previous cell (or a failed run) left plain directories at the mountpoint.
    shutil.rmtree(DRIVE_ROOT)
drive.mount(str(DRIVE_ROOT))
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

if not MAIN.exists():
    run(["git", "clone", "https://github.com/SharvaGogawale1/concept-aware-training.git", MAIN])
else:
    run(["git", "-C", str(MAIN), "pull", "--ff-only"])
if not EXT.exists():
    run(["git", "clone", "https://github.com/christine-zhang1/learning-concepts.git", EXT])
run(["git", "checkout", "--detach", UPSTREAM_COMMIT], cwd=EXT)
patch_file = MAIN / "external" / "learning-concepts.patch"
assert patch_file.exists(), "Commit external/learning-concepts.patch before running Colab."
# Reset to the pinned commit and wipe every patch artefact before re-applying.
# Testing "does it apply, else does it reverse-apply" only held while the patch
# never changed: once a newer patch lands, the old one is applied, neither
# direction matches, and the run dies on an assertion.  Resetting is idempotent.
# EXT holds upstream code only -- the corpus lives in DATA -- so clean is safe.
run(["git", "-C", str(EXT), "reset", "--hard", UPSTREAM_COMMIT])
run(["git", "-C", str(EXT), "clean", "-fdq"])
run(["git", "apply", str(patch_file)], cwd=EXT)
print("patch applied onto", UPSTREAM_COMMIT[:7])

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(EXT), "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q", "accelerate",
     "peft", "bitsandbytes", "datasets", "spacy", "mteb>=1.12", "wandb",
     "nltk", "scipy", "scikit-learn", "seaborn", "pytest"])
# Pinned LAST so nothing above can pull it forward.  Upstream pins no versions,
# but their extraction reuses a prefix KV cache through
# DynamicCache.from_legacy_cache, which transformers removed in v5; the current
# Colab image installs v5 and the first shard dies with AttributeError.  The
# 4.5x line keeps that API and still satisfies our Task-14 evaluators.
run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51,<4.58"])
# Colab ships torchao 0.10, but peft requires >=0.16 and RAISES from
# is_torchao_available() rather than degrading.  That call sits inside
# PeftModel.from_pretrained, which every evaluator uses to load an adapter, so
# without this the failure lands hours later at evaluation rather than here.
# get_peft_model takes a different path, which is why training itself succeeds.
run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16"])
import transformers as _tf
print("transformers", _tf.__version__)
run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
run([sys.executable, MAIN / "builddataset/verify_task14_data.py",
     "--repo_root", MAIN, "--download_missing",
     "--report_json", DRIVE_RESULTS / "external_benchmark_integrity.json"], cwd=MAIN)
(DRIVE_RESULTS / "environment_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)

from huggingface_hub import login
login(token=getpass("Hugging Face token (input hidden): "), add_to_git_credential=False)


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
# /content is wiped between sessions; pull the generated concept data back rather
# than paying the multi-hour regeneration again.
restore_dataset_from_drive(LEAF)

def adapter_path(method, seed, value):
    path = RUNS / MODEL_TAG / method / f"seed_{seed}" / str(value)
    assert_ephemeral(path)
    return path

def finished(path):
    """A run counts as finished when its adapter exists locally or on Drive."""
    if (Path(path) / "adapter_config.json").is_file():
        return True
    return restore_adapter_from_drive(path)

def train_flat(method, seed, concept_weight, *, objective="set_marginal",
               slot_ntp_weight=None, contrast_beta=0.0, exclude_target=False,
               randomized=False, data_augmentation=False, epochs=5, train_file=None,
               max_samples=None, batch=8, accum=2, alt_aux="none", alt_aux_weight=0.0):
    # The aux suffix is added only when the term is on, so every adapter trained
    # before it existed keeps its path and still resumes.
    aux_tag = "" if alt_aux == "none" else f"_aux_{alt_aux}_{alt_aux_weight}"
    out = adapter_path(method, seed, f"lambda_{concept_weight}_beta_{contrast_beta}{aux_tag}")
    args = [sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
            "--dataset-type", "embedding", "--concept-loss-weight", concept_weight,
            "--concept-objective", objective, "--contrast-beta", contrast_beta,
            "--seed", seed, "--num-train-epochs", epochs, "--output-dir", out,
            "--save-strategy", "no", "--report-to", "none",
            "--per-device-train-batch-size", batch,
            "--gradient-accumulation-steps", accum]
    if slot_ntp_weight is not None: args += ["--slot-ntp-weight", slot_ntp_weight]
    if alt_aux != "none": args += ["--alt-aux", alt_aux, "--alt-aux-weight", alt_aux_weight]
    # Drops the observed target from the concept set, so the loss cannot be paid
    # with the mass NTP already put there.  Pair it with slot_ntp_weight=1.0 or
    # the observed target is pushed down.  adapter_path() does not encode this
    # flag, so the METHOD name must differ from the target-inclusive arm's.
    if exclude_target: args += ["--exclude-target"]
    if randomized: args += ["--randomized-synonyms"]
    if data_augmentation: args += ["--use-data-augmentation"]
    if train_file: args += ["--train-file", train_file]
    if max_samples: args += ["--max-train-samples", max_samples]
    # Released effective batch is 8 x 2 = 16; it is recorded in every config.
    if RESUME_FINISHED_RUNS and finished(out):
        print("resume: already trained, skipping", out)
        return out
    run(args, cwd=EXT)
    cache_adapter_to_drive(out)
    return out


## Build conservative negatives, and a 50-row sample to read by hand

Candidates must occur in the model’s top-100 next-token pool, match POS, lie outside the alternative set, share no WordNet synset with any alternative, not be a morphological variant, and fall below the contextual-similarity ceiling. Coverage and every rejection reason are reported. The 50-row sample is an error analysis, not an annotation project: read it before trusting any contrastive number.


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
NEG_TRAIN = LEAF / "synonyms_train_conservative_negatives.jsonl"
SCREEN_DIR = DRIVE_RESULTS / f"task15b_screen{_TAG_SUFFIX}"
SCREEN_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DATA:
    run([sys.executable, "data/build_contrastive_negatives.py",
         "--source", LEAF / "synonyms_train.jsonl",
         "--topk", DATA / "c4" / MODEL_TAG / "prompting" / "topk_*.jsonl",
         "--output", NEG_TRAIN,
         # Per model: a shared name would let the 3B pass overwrite the 1B report.
         "--report", DRIVE_RESULTS / f"contrastive_negative_report{_TAG_SUFFIX}.json",
         "--max-cosine", "0.35", "--max-negatives", "20"], cwd=EXT)
    cache_dataset_to_drive(LEAF)
    # Stratified by POS and alternative-set size so the sample cannot be all easy
    # nouns with large sets.  Fixed seed: the same 50 rows every time it is rerun.
    import csv, random
    buckets = {}
    with NEG_TRAIN.open(encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            for target in row.get("content_word_responses", []):
                if not target.get("negatives"):
                    continue
                key = (target.get("pos") or "?", "small" if len(target.get("synonyms", [])) <= 2 else "large")
                buckets.setdefault(key, []).append({
                    "pos": key[0], "set_size": key[1], "context": row["input_sequence"],
                    "observed_target": target["word"],
                    "alternatives": " | ".join(target.get("synonyms", [])),
                    "negatives": " | ".join(target["negatives"]),
                    "negative_scores": " | ".join(f"{s:.3f}" for s in target.get("negative_scores", []))})
    rng = random.Random(42)
    picked, per_bucket = [], max(1, 50 // max(1, len(buckets)))
    for key in sorted(buckets):
        picked += rng.sample(buckets[key], min(per_bucket, len(buckets[key])))
    remainder = [item for key in sorted(buckets) for item in buckets[key] if item not in picked]
    picked += rng.sample(remainder, min(50 - len(picked), len(remainder)))
    sample_path = SCREEN_DIR / "negative_sample_50.csv"
    with sample_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(picked[0].keys()) if picked else ["context"])
        writer.writeheader(); writer.writerows(picked)
    print(f"wrote {len(picked)} rows across {len(buckets)} strata to", sample_path)


## Screen on seed 42

$\alpha\in\{.25,.5,1\}$ controls concept pressure on the alternative-only set; the concept-slot NTP weight is 1 in every arm, so the observed target is always trained. Uniform (mean-log) losses are roughly an order of magnitude larger per slot than set-marginal at the same weight, which is why this grid starts lower than Zhang's.

$\alpha$ is selected on **C4 validation** (lowest alternative NLL subject to the global-NLL, observed-target-NLL and collapse gates printed by the decision cell). Only then is one target-inclusive ablation trained at that $\alpha$, and $\beta\in\{.25,.5,1\}$ screened on **SWORDS dev**. SWORDS test is never read during tuning.


In [ ]:
OBJECTIVE_RUNS = load_runs(f"task15b{_TAG_SUFFIX}")
if RUN_SCREEN:
    for alpha in [0.25, 0.5, 1.0]:
        OBJECTIVE_RUNS[f"alternative_uniform_alpha{alpha}_seed42"] = train_flat(
            "alternative_uniform", 42, alpha, objective="uniform", slot_ntp_weight=1.0,
            exclude_target=True)

# Set this ONLY from the alpha gate printed by the decision cell below -- or,
# when REPLICATING on another model, to the value that gate already selected
# elsewhere, passed in as SELECTED_ALPHA=0.5 rather than edited in here.  A
# transferred value is a deliberate choice and belongs in the write-up: it means
# alpha was not tuned on this model, which is the stronger claim and also the
# only honest one once the test set has been read for the first model.
SELECTED_ALPHA = float(os.environ["SELECTED_ALPHA"]) if os.environ.get("SELECTED_ALPHA") else None
if RUN_SCREEN and SELECTED_ALPHA is not None:
    # The one ablation that isolates the exclusion: identical loss, observed
    # target back inside the concept set.  Not a headline method.
    OBJECTIVE_RUNS[f"inclusive_uniform_alpha{SELECTED_ALPHA}_seed42"] = train_flat(
        "inclusive_uniform", 42, SELECTED_ALPHA, objective="uniform", slot_ntp_weight=1.0)
    for beta in [0.25, 0.5, 1.0]:
        OBJECTIVE_RUNS[f"contrast_alpha{SELECTED_ALPHA}_beta{beta}_seed42"] = train_flat(
            "alternative_contrastive", 42, SELECTED_ALPHA, objective="uniform",
            slot_ntp_weight=1.0, exclude_target=True, contrast_beta=beta, train_file=NEG_TRAIN)
    # Frontier point.  Contrastive arms land between alpha=.5 and alpha=1 on BOTH
    # axes, so two uniform points cannot say whether they sit above the alpha
    # curve or merely on it.  Not an alpha-selection candidate: alpha was chosen
    # on validation from {.25,.5,1} before this ran, and this is scored after.
    OBJECTIVE_RUNS["alternative_uniform_alpha0.75_seed42"] = train_flat(
        "alternative_uniform", 42, 0.75, objective="uniform", slot_ntp_weight=1.0,
        exclude_target=True)


## Calibrated concept marginalization, and what the negatives were really doing

**Why.** Decompose the SWORDS ranking gain. GAP rises either by pushing rejected candidates down or by pulling accepted ones up, and set-marginal training does only the first. Across Llama-1B (3 seeds), Llama-3B and Qwen3-1.7B it lowers rejected-mass share every time (−.014, −.014, −.021) and never lowers NLL on human-accepted alternatives (−.012 n.s., −.046 n.s., **+.088 worse**). Alternative-only supervision moves precisely that second axis, by the same amount in both families (−.50 nats on Llama, −.485 [−.568,−.403] on Qwen). So the two objectives are not rivals; one contains the other. With $q = p / P(S)$ the model's own distribution inside the set,

$$-\tfrac{1}{n}\sum_{c\in S}\log p_c \;=\; \underbrace{-\log P(S)}_{\text{Zhang}} \;+\; \underbrace{\mathrm{KL}(u\,\|\,q)}_{\text{within-set}} \;+\; \log n ,$$

and $\nabla_z[-\log P(S)] = p - q\,\mathbb{1}_S$: the set marginal is self-distillation toward the model's current within-set distribution, so nothing in it says *which* members deserve mass. A semantically randomized control is a second, weaker probe of the same point: on Llama it reproduces most of the ranking gain (+.008 of Zhang's +.011), on Qwen it reproduces none (−.001). That comparison is therefore **model-dependent and reported as such**; the decomposition above is what replicates. These arms keep Zhang's term exactly as released (target-inclusive, $\lambda=1$, 0.6 cutoff) and add one term on the alternatives only:

- `uniform` — $-\tfrac1n\sum\log p_a$: raises the alternatives and spreads them.
- `within_kl` — $\mathrm{KL}(u\|q_A)$ alone. Its logit gradient is zero outside the alternatives and sums to zero inside them, so it moves mass *between* alternatives and leaves the observed word and the set's total mass to Zhang's term. If SWORDS moves under this arm, the gain is within-set calibration; if only `uniform` moves it, the gain is mass, and the decomposition says so.

**Gate, fixed before any of these is scored (SWORDS dev only):** STS $\ge .5469$; GAP and AUROC above Zhang with paired intervals excluding zero; above randomized $\lambda=.25$ on GAP and AUROC; global NLL $\le$ NTP $+.20$; observed-target NLL $\le$ NTP $+.10$. The smallest weight that passes is the method. If none passes, the result is the STS-vs-GAP frontier these arms trace, reported as a trade-off.

**The negatives.** A hand read of `negative_sample_50.csv` (2026-09-18): the contrastive loss only fires where a slot has an alternative (31/50), and in 23 of those 31 every negative is a letter or a word-prefix token; 2 of 31 have a semantically meaningful negative. The 0.35 similarity ceiling rejects every real word in a slot with rich alternatives, so only non-words survive, and WordNet lists them. Two controls settle what the published +.005 GAP was: `clean` (complete words in their dominant POS, plus antonyms of the observed word) and `fragments` (only what `clean` throws away). Effective coverage — slots with an alternative AND a negative — is printed for both; the raw 45.6% is not the supervised fraction.


In [ ]:
NEG_VARIANTS = {"clean": ["--strict-lexical", "--antonyms"], "fragments": ["--fragments-only"]}
NEG_FILES = {name: LEAF / f"synonyms_train_negatives_{name}.jsonl" for name in NEG_VARIANTS}
HYBRID_GRID = [("uniform", 0.25), ("uniform", 0.5), ("within_kl", 0.25), ("within_kl", 0.5), ("within_kl", 1.0)]
# "within_kl:0.5" -- set ONLY from the gate above, then rerun with RUN_MULTISEED.
SELECTED_HYBRID = os.environ.get("SELECTED_HYBRID")
if RUN_HYBRID:
    for kind, weight in HYBRID_GRID:
        OBJECTIVE_RUNS[f"zhang_plus_{kind}_g{weight}_seed42"] = train_flat(
            "zhang_plus_aux", 42, 1.0, alt_aux=kind, alt_aux_weight=weight)
    # Confirmation of the arm the gate below selected.  RUN_CONFIRM is NOT used
    # for this: that flag also relaunches the alternative-uniform and legacy
    # contrastive arms, which are ablations here, not the method.
    if SELECTED_HYBRID:
        kind, weight = SELECTED_HYBRID.split(":"); weight = float(weight)
        assert (kind, weight) in HYBRID_GRID, "SELECTED_HYBRID must be one of the screened arms"
        for seed in SEEDS:
            OBJECTIVE_RUNS[f"calibrated_seed{seed}"] = train_flat(
                "zhang_plus_aux", seed, 1.0, alt_aux=kind, alt_aux_weight=weight)
        # Same adapter as the seed-42 screen arm; one key per adapter.
        OBJECTIVE_RUNS.pop(f"zhang_plus_{kind}_g{weight}_seed42", None)

if RUN_NEGATIVE_CONTROLS:
    topk_glob = DATA / "c4" / MODEL_TAG / "prompting" / "topk_*.jsonl"
    if not list(topk_glob.parent.glob(topk_glob.name)):
        print("no top-k shards restored; the negative controls are skipped, not faked")
    else:
        assert SELECTED_ALPHA is not None, "the negative controls reuse the locked alpha"
        for name, flags in NEG_VARIANTS.items():
            report = DRIVE_RESULTS / f"contrastive_negative_report_{name}{_TAG_SUFFIX}.json"
            if not NEG_FILES[name].is_file():
                run([sys.executable, "data/build_contrastive_negatives.py",
                     "--source", LEAF / "synonyms_train.jsonl", "--topk", topk_glob,
                     "--output", NEG_FILES[name], "--report", report,
                     "--max-cosine", "0.35", "--max-negatives", "20",
                     # Coverage counted the way the TRAINER counts: a multi-token
                     # candidate never reaches the loss, so the raw fraction
                     # overstates what is actually supervised.
                     "--tokenizer", BASE_MODEL, *flags], cwd=EXT)
            if report.is_file():
                stats = json.loads(report.read_text())
                print(f"{name}: raw coverage {stats['negative_coverage']:.3f}, "
                      f"EFFECTIVE contrastive coverage {stats['effective_contrastive_coverage']:.3f}")
            # A distinct method name per file: adapter_path() does not see train_file.
            OBJECTIVE_RUNS[f"contrast_{name}_negatives_seed42"] = train_flat(
                f"alternative_contrastive_{name}", 42, SELECTED_ALPHA, objective="uniform",
                slot_ntp_weight=1.0, exclude_target=True, contrast_beta=1.0,
                train_file=NEG_FILES[name])

if RUN_HYBRID or RUN_NEGATIVE_CONTROLS:
    save_runs(OBJECTIVE_RUNS, f"task15b{_TAG_SUFFIX}")


## Locked confirmation

Lock $\alpha$ and $\beta$ from the seed-42 screen's decision cell further down; never choose them per seed. This cell sits before evaluation on purpose, so one Run All trains the new seeds and then scores them. Seeds 42, 123, 2024 for the alternative-only uniform arm and, if promoted, the contrastive arm. The seed-42 adapters already exist and resume for free.


In [ ]:
SELECTED_BETA = float(os.environ["SELECTED_BETA"]) if os.environ.get("SELECTED_BETA") else None
if RUN_CONFIRM:
    assert SELECTED_ALPHA is not None, "set SELECTED_ALPHA from the alpha gate first"
    for seed in SEEDS:
        OBJECTIVE_RUNS[f"alternative_uniform_seed{seed}"] = train_flat(
            "alternative_uniform", seed, SELECTED_ALPHA, objective="uniform", slot_ntp_weight=1.0,
            exclude_target=True)
        if SELECTED_BETA is not None:
            OBJECTIVE_RUNS[f"contrastive_seed{seed}"] = train_flat(
                "alternative_contrastive", seed, SELECTED_ALPHA, objective="uniform",
                slot_ntp_weight=1.0, exclude_target=True, contrast_beta=SELECTED_BETA, train_file=NEG_TRAIN)
    # The screen already trained seed 42 at the locked values and adapter_path()
    # maps both calls to one directory; two keys on one adapter would score it twice.
    OBJECTIVE_RUNS.pop(f"alternative_uniform_alpha{SELECTED_ALPHA}_seed42", None)
    if SELECTED_BETA is not None:
        OBJECTIVE_RUNS.pop(f"contrast_alpha{SELECTED_ALPHA}_beta{SELECTED_BETA}_seed42", None)
    for label, path in OBJECTIVE_RUNS.items(): sync_small_artifacts(path, f"task15b_logs{_TAG_SUFFIX}/{label}")
    audit_no_drive_weights()
save_runs(OBJECTIVE_RUNS, f"task15b{_TAG_SUFFIX}")


## Evaluation

Every checkpoint of this pass — Task 15's arms and this notebook's — is scored by the same evaluators. A validation pass of the perplexity and concept-set evaluators exists only for choosing $\alpha$; every other number is C4 test, SWORDS dev, the nine STS tasks and bm-semlex.


In [ ]:
if RUN_EVAL:
    OBJECTIVE_RUNS = restore_all(OBJECTIVE_RUNS)
    # The question is "does this beat Zhang", so Zhang's arms sit IN this table:
    # pretrained as the reference row, NTP and augmented NTP as matched controls,
    # randomized at both weights as the semantic control, and set-marginal at
    # lambda=1 as the method being improved on.  They come from Task 15's manifest;
    # an arm Task 15 never trained is reported as absent, never retrained here.
    task15_runs = restore_all(load_runs(f"task15{_TAG_SUFFIX}"))
    BASELINE_LABELS = ("ntp_seed42", "augmented_ntp_seed42", "randomized_seed42",
                       "randomized_lambda1.0_seed42", "zhang_seed42", "zhang_lambda1.0_seed42")
    baseline_runs = {label: task15_runs[label] for label in BASELINE_LABELS if label in task15_runs}
    # RUN_CONFIRM in Task 15 pops zhang_lambda1.0_seed42 in favour of zhang_seed42;
    # both name ONE adapter, so keep whichever exists and never both.
    if "zhang_seed42" in baseline_runs:
        baseline_runs.pop("zhang_lambda1.0_seed42", None)
    for label in BASELINE_LABELS[:-1]:
        if label not in baseline_runs and not (label == "zhang_seed42" and "zhang_lambda1.0_seed42" in baseline_runs):
            print("Task 15 never trained this baseline; the table will lack it:", label)
    all_runs = {**baseline_runs, **OBJECTIVE_RUNS}
    checkpoints = [BASE_MODEL, *map(str, all_runs.values())]
    result_dir = SCREEN_DIR
    # Each evaluator is skipped only when its own output already scores every
    # checkpoint of this pass, so a disconnect costs at most one evaluator.
    # VALIDATION pass first: this is the only thing the alpha choice may read.
    guarded(result_dir / "val_perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_val.jsonl",
             "--output", result_dir / "val_perplexity.json"], EXT, "validation perplexity")
    guarded(result_dir / "val_concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_val.jsonl",
             "--output", result_dir / "val_concept_sets.json"], EXT, "validation concept sets")
    guarded(result_dir / "perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "perplexity.json"], EXT, "perplexity")
    guarded(result_dir / "concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "concept_sets.json"], EXT, "concept sets")
    task15_dir = DRIVE_RESULTS / RESULT_DIR
    for label, checkpoint in {"pretrained": BASE_MODEL, **all_runs}.items():
        display_label = label.replace("_", " ")
        csv_output = result_dir / f"sts_{display_label}.csv"
        # STS is deterministic per checkpoint and Task 15 already scored the
        # baselines, so reuse its file rather than spending 3.5 min re-deriving it.
        previous = task15_dir / csv_output.name
        if not csv_output.is_file() and previous.is_file():
            shutil.copy2(previous, csv_output)
        if sts_covered(csv_output):
            print("resume: STS already scored, skipping", display_label)
            continue
        mteb_args = [sys.executable, "eval/eval_mteb.py", "--base-model", BASE_MODEL,
                     "--dataset", "c4", "--dataset-type", "embedding", "--tasks", "sts",
                     "--run-label", display_label, "--csv-output", csv_output,
                     "--mteb-output-root", result_dir / "mteb_raw"]
        mteb_args += ["--no-adapter"] if checkpoint == BASE_MODEL else ["--adapter-path", checkpoint]
        run(mteb_args, cwd=EXT)
    guarded(result_dir / "swords.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--swords_json", MAIN / "data/swords/swords-v1.1_dev.json.gz",
             "--results_json", result_dir / "swords.json", "--modes", "left", "full"], MAIN, "SWORDS")
    # One paired interval per reference, looked up by label so an index can never
    # drift onto the wrong arm.  vs zhang is the headline; vs randomized is what
    # says whether a gain is semantic rather than distributional; vs the selected
    # alternative-uniform arm is the only fair test of the contrastive term itself.
    references = {"pretrained": BASE_MODEL, **{label: str(path) for label, path in baseline_runs.items()}}
    # EVERY alternative-only uniform arm is a reference, not just the selected one.
    # Contrastive has to beat the cheaper way of buying the same concept pressure,
    # which is turning alpha up.  A same-alpha comparison alone cannot show that:
    # it credits the negatives with a gain a larger alpha also delivers, which is
    # the identical error this paper accuses set-marginal training of.
    # Includes the confirmation arms (alternative_uniform_seed123, ...), so each
    # contrastive seed has a paired interval against the uniform arm of the SAME
    # seed -- the one comparison that isolates the contrastive term across seeds.
    for key, path in OBJECTIVE_RUNS.items():
        if key.startswith("alternative_uniform"):
            references[key] = str(path)
    for label, reference in references.items():
        run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
             "--kind", "swords", "--results-json", result_dir / "swords.json",
             "--baseline-index", checkpoints.index(reference),
             "--output", result_dir / f"swords_paired_ci_vs_{label}.json"], cwd=MAIN)
    guarded(result_dir / "bm_semlex.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_bm_semlex.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--data", MAIN / "data/bm_semlex/curated_200.tsv",
             "--results_json", result_dir / "bm_semlex.json"], MAIN, "bm-semlex")
    manifest = {"pretrained": BASE_MODEL, **{label.replace("_", " "): str(path) for label, path in all_runs.items()}}
    (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    run([sys.executable, MAIN / "scripts/summarize_concept_experiments.py",
         "--manifest", result_dir / "manifest.json", "--result-dir", result_dir,
         "--output", result_dir / "flat_extension_table.csv"], cwd=MAIN)


## Decision cell

Reads only what the evaluation cell wrote and prints every gate with its number, so the choice is auditable from the notebook output alone. First pass: the $\alpha$ gate (validation only). After `SELECTED_ALPHA` is set and the $\beta$ arms are scored: the promotion gates and the headline.

Headline rule: contrastive if it passes every promotion gate; otherwise alternative-only uniform if it passes the same gates minus the vs-itself clause; otherwise this is a negative result and is reported as one.


In [ ]:
import csv
GLOBAL_NLL_SLACK, OBSERVED_NLL_SLACK, STS_SLACK, BM_SLACK, MIN_PROB_RATIO = 0.20, 0.10, 0.005, 0.02, 0.5
result_dir = SCREEN_DIR

def _by_ckpt(name):
    path = result_dir / name
    return {str(r["checkpoint"]): r for r in json.loads(path.read_text())} if path.is_file() else {}

def _sts(label):
    path = result_dir / f"sts_{label.replace('_', ' ')}.csv"
    if not path.is_file():
        return None
    with path.open(encoding="utf-8") as handle:
        scores = [float(r["main_score"]) for r in csv.DictReader(handle)]
    return sum(scores) / len(scores) if scores else None

def _paired(reference_label, candidate):
    path = result_dir / f"swords_paired_ci_vs_{reference_label}.json"
    if not path.is_file():
        return {}
    for entry in json.loads(path.read_text()):
        if str(entry["candidate"]) == str(candidate):
            return entry["metrics"]
    return {}

def _sig(metrics, key, better):
    # (candidate - reference, True when the 95% CI excludes zero on the good side)
    value = metrics.get(key)
    if not value or value.get("candidate_minus_baseline") is None:
        return None, None
    lo, hi = value["ci95"]
    return value["candidate_minus_baseline"], (lo > 0) if better == "up" else (hi < 0)

def _fmt(x):
    return "n/a" if x is None else f"{x:.4f}"

manifest_path = result_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
runs = {label.replace(" ", "_"): str(path) for label, path in manifest.items()}
ntp = runs.get("ntp_seed42")
zhang_label = "zhang_seed42" if "zhang_seed42" in runs else "zhang_lambda1.0_seed42"
zhang, randomized = runs.get(zhang_label), runs.get("randomized_seed42")
val_c, val_p = _by_ckpt("val_concept_sets.json"), _by_ckpt("val_perplexity.json")
tst_p, bm = _by_ckpt("perplexity.json"), _by_ckpt("bm_semlex.json")

print("== alpha gate: C4 VALIDATION only ==")
g_ntp = val_p.get(ntp, {}).get("global", {}).get("mean_nll")
# The observed target is referenced against NTP, never against Zhang.  Zhang's
# supervised set CONTAINS the observed token, so its objective actively drives
# that NLL below NTP's; asking an alternative-only arm to match it would be
# asking it to do the one thing it exists not to do.  NTP is the matched control,
# and the question here is only "is the observed target damaged?" -- which
# --slot-ntp-weight 1.0 is what prevents.
o_ntp = val_c.get(ntp, {}).get("observed_target_nll")
mp_zhang = val_c.get(zhang, {}).get("minimum_candidate_probability")
alpha_rows = []
for label, ck in runs.items():
    if not label.startswith("alternative_uniform_alpha"):
        continue
    c, p = val_c.get(ck, {}), val_p.get(ck, {})
    alt, g = c.get("alternative_nll"), p.get("global", {}).get("mean_nll")
    o, mp = c.get("observed_target_nll"), c.get("minimum_candidate_probability")
    gates = {"global<=ntp+.20": None if None in (g, g_ntp) else g <= g_ntp + GLOBAL_NLL_SLACK,
             "observed<=ntp+.10": None if None in (o, o_ntp) else o <= o_ntp + OBSERVED_NLL_SLACK,
             "min_prob>=.5*zhang": None if None in (mp, mp_zhang) else mp >= MIN_PROB_RATIO * mp_zhang}
    alpha_rows.append((label, alt, gates))
    print(f"  {label:38} alt_nll={_fmt(alt)} global={_fmt(g)} observed={_fmt(o)} min_prob={_fmt(mp)}  "
          + "  ".join(f"{k}:{'n/a' if v is None else ('PASS' if v else 'FAIL')}" for k, v in gates.items()))
passing = [r for r in alpha_rows if r[1] is not None and all(r[2].values())]
if passing:
    best = min(passing, key=lambda r: r[1])
    print("  -> SELECTED_ALPHA =", best[0].split("alpha", 1)[1].split("_", 1)[0])
elif alpha_rows:
    print("  -> no alpha passes every gate.  Report that; do not loosen the gates.")
else:
    print("  (no alternative-uniform arms scored yet)")

print("== promotion gates: SWORDS DEV paired intervals, C4 test, STS, bm-semlex ==")
g_ntp_test = tst_p.get(ntp, {}).get("global", {}).get("mean_nll")
sts_zhang = _sts(zhang_label)
acc_zhang = bm.get(zhang, {}).get("left", {}).get("accuracy")

def promotion(label, ck, self_label=None):
    vz, vr = _paired(zhang_label, ck), _paired("randomized_seed42", ck)
    vs = _paired(self_label, ck) if self_label else {}
    g = tst_p.get(ck, {}).get("global", {}).get("mean_nll")
    sts, acc = _sts(label), bm.get(ck, {}).get("left", {}).get("accuracy")
    d_gap_z, s_gap_z = _sig(vz, "gap", "up")
    d_auroc_s, s_auroc_s = _sig(vs, "auroc", "up")
    d_gap_r, _ = _sig(vr, "gap", "up")
    _, s_auroc_r = _sig(vr, "auroc", "up")
    _, s_rms_r = _sig(vr, "rejected_mass_share", "down")
    d_alt_z, s_alt_z = _sig(vz, "alternatives_nll", "down")
    # Frontier gate.  An arm is DOMINATED when some uniform arm already matches or
    # beats its ranking at no greater language-modelling cost -- that arm's gain is
    # the alpha curve, not the negatives.  Uniform arms that cost MORE global NLL
    # are excluded: losing to a costlier arm is a trade, not a domination.
    dominated = []
    for u_label, u_ck in (runs.items() if self_label else ()):
        u_global = tst_p.get(u_ck, {}).get("global", {}).get("mean_nll")
        if not u_label.startswith("alternative_uniform_alpha") or str(u_ck) == str(ck):
            continue
        if None in (u_global, g) or u_global > g:
            continue
        _, beats = _sig(_paired(u_label, ck), "gap", "up")
        if beats is not None:
            dominated.append(not beats)
    on_frontier = None if not dominated else not any(dominated)
    # Two groups, reported and decided separately.  DISCRIMINATION is the claim:
    # does this arm rank human-labelled substitutes better than Zhang, and better
    # than the randomized control Zhang cannot separate from?  RETENTION is the
    # price: are STS and language modelling preserved?  Collapsing them into one
    # verdict turns "wins the claim, pays on STS" -- a reportable trade-off and
    # the most likely real outcome -- into the same output as "nothing worked".
    discrimination = {
        f"GAP > zhang (CI excl 0)  [{_fmt(d_gap_z)}]": None if s_gap_z is None else bool(s_gap_z),
        f"AUROC > same non-contrastive arm (CI excl 0)  [{_fmt(d_auroc_s)}]": (None if not self_label or s_auroc_s is None else bool(s_auroc_s)),
        f"GAP >= randomized lambda.25  [{_fmt(d_gap_r)}]": None if d_gap_r is None else d_gap_r >= 0,
        "AUROC or rejected-mass share better than randomized (CI excl 0)": (None if s_auroc_r is None and s_rms_r is None else bool(s_auroc_r or s_rms_r)),
        f"human-accepted alt NLL < zhang (CI excl 0)  [{_fmt(d_alt_z)}]": None if s_alt_z is None else bool(s_alt_z),
        "GAP > every uniform arm costing no more global NLL (CI excl 0)": on_frontier,
    }
    retention = {
        f"global NLL <= ntp+.20  [{_fmt(g)} vs {_fmt(g_ntp_test)}]": None if None in (g, g_ntp_test) else g <= g_ntp_test + GLOBAL_NLL_SLACK,
        f"STS >= zhang-.005  [{_fmt(sts)} vs {_fmt(sts_zhang)}]": None if None in (sts, sts_zhang) else sts >= sts_zhang - STS_SLACK,
        f"bm-semlex >= zhang-2pp  [{_fmt(acc)} vs {_fmt(acc_zhang)}]": None if None in (acc, acc_zhang) else acc >= acc_zhang - BM_SLACK,
    }
    print(f"  {label}")
    for heading, group in (("discrimination (the claim)", discrimination), ("retention (the price)", retention)):
        print(f"    {heading}")
        for name, ok in group.items():
            print(f"      {'n/a ' if ok is None else ('PASS' if ok else 'FAIL')}  {name}")
    def _all(group):
        known = [ok for ok in group.values() if ok is not None]
        return bool(known) and all(known)
    return _all(discrimination), _all(retention)

winners = {"contrastive": [], "alternative_uniform": []}
retained = {}
for label, ck in runs.items():
    if label.startswith("contrast_alpha"):
        alpha = label.split("alpha", 1)[1].split("_", 1)[0]
        discriminates, retains = promotion(label, ck, self_label=f"alternative_uniform_alpha{alpha}_seed42")
        if discriminates:
            winners["contrastive"].append(label)
        retained[label] = retains
    elif label.startswith("alternative_uniform_alpha"):
        discriminates, retains = promotion(label, ck)
        if discriminates:
            winners["alternative_uniform"].append(label)
        retained[label] = retains

def announce(kind, labels):
    for label in labels:
        price = ("and pays nothing: every retention gate holds" if retained.get(label)
                 else "but FAILS a retention gate above -- that trade-off is a result, report it")
        print(f"  {kind} wins every discrimination gate: {label} {price}")

print("== headline ==")
if winners["contrastive"]:
    announce("contrastive", winners["contrastive"])
    print("  -> beta is TUNED on SWORDS dev (test is never read here), so any passing")
    print("     beta is admissible.  Take the smallest unless the frontier margin is")
    print("     monotone in beta, in which case take the largest that still retains,")
    print("     and record the choice and its reason before the seeds are run.")
elif winners["alternative_uniform"]:
    announce("alternative-only uniform", winners["alternative_uniform"])
    print("  -> contrastive did not separate from it; the uniform arm is the headline candidate")
else:
    print("  no arm wins every discrimination gate: this is a negative result and is reported as one")


## Hybrid gate (automatic)

Applied to the H1/H2 arms only, on SWORDS **dev**, and evaluated before any of these numbers is read by hand. Thresholds are derived from this notebook's own Zhang and NTP rows rather than hardcoded, so the same cell gates a second model without edits.

An arm passes only if it beats Zhang on the axis Zhang provably does not move (human-accepted alternative NLL) **and** on ranking, stays within the retention allowances, and also beats the randomized control. The smallest weight that passes is selected; ties go to the smaller weight. If nothing passes, that is the result and the paper reports the trade-off curve — the cell does not relax a threshold to manufacture a winner.


In [ ]:
STS_ALLOWANCE = 0.005        # vs Zhang
GLOBAL_NLL_ALLOWANCE = 0.20  # vs NTP
OBSERVED_NLL_ALLOWANCE = 0.10

def _gate_rows():
    table = SCREEN_DIR / "flat_extension_table.csv"
    if not table.is_file():
        print("no flat_extension_table.csv yet; run RUN_EVAL first"); return None
    import csv
    return {r["method"]: r for r in csv.DictReader(table.open())}

def _ci(path, run_path, metric):
    # Returns (delta, significant) for one candidate in one paired-CI file.
    if not Path(path).is_file():
        return None
    for entry in json.loads(Path(path).read_text()):
        if str(entry["candidate"]) != str(run_path):
            continue
        if metric not in entry["metrics"]:
            return None
        d = entry["metrics"][metric]
        low, high = d["ci95"]
        return d["candidate_minus_baseline"], (low * high > 0)
    return None

def hybrid_gate(verbose=True):
    rows = _gate_rows()
    if rows is None: return None
    zhang = rows.get("zhang seed42") or rows.get("zhang lambda1.0 seed42")
    ntp = rows.get("ntp seed42")
    if not zhang or not ntp:
        print("Task 15 baselines missing from the table; cannot gate"); return None
    sts_floor = float(zhang["sts_mean"]) - STS_ALLOWANCE
    nll_ceiling = float(ntp["global_nll"]) + GLOBAL_NLL_ALLOWANCE
    obs_ceiling = float(ntp["swords_observed_target_nll"]) + OBSERVED_NLL_ALLOWANCE
    print(f"thresholds -> STS >= {sts_floor:.4f} | global NLL <= {nll_ceiling:.4f} "
          f"| observed-target NLL <= {obs_ceiling:.4f}")

    vs_zhang = SCREEN_DIR / ("swords_paired_ci_vs_zhang_seed42.json"
                             if (SCREEN_DIR / "swords_paired_ci_vs_zhang_seed42.json").is_file()
                             else "swords_paired_ci_vs_zhang_lambda1.0_seed42.json")
    vs_rand = SCREEN_DIR / "swords_paired_ci_vs_randomized_seed42.json"

    passing = []
    for kind, weight in HYBRID_GRID:
        label = f"zhang_plus_{kind}_g{weight}_seed42"
        run_path = OBJECTIVE_RUNS.get(label)
        row = rows.get(label.replace("_", " "))
        if row is None or run_path is None:
            if verbose: print(f"  {label:34} not trained/scored yet")
            continue
        checks = {
            "STS": float(row["sts_mean"]) >= sts_floor,
            "globalNLL": float(row["global_nll"]) <= nll_ceiling,
            "obsNLL": float(row["swords_observed_target_nll"]) <= obs_ceiling,
        }
        for metric, key, want_negative in (("alternatives_nll", "altNLL<Zhang", True),
                                           ("gap", "GAP>Zhang", False),
                                           ("auroc", "AUROC>Zhang", False)):
            got = _ci(vs_zhang, run_path, metric)
            checks[key] = bool(got and got[1] and ((got[0] < 0) == want_negative))
        got = _ci(vs_rand, run_path, "gap")
        checks["GAP>random"] = bool(got and got[1] and got[0] > 0)

        ok = all(checks.values())
        if verbose:
            failed = [k for k, v in checks.items() if not v]
            print(f"  {label:34} {'PASS' if ok else 'FAIL'}"
                  f"  STS {float(row['sts_mean']):.4f}"
                  f"  gNLL {float(row['global_nll']):.4f}"
                  f"  altNLL {float(row['swords_alternative_nll']):.3f}"
                  + ("" if ok else f"   failed: {', '.join(failed)}"))
        if ok: passing.append((weight, kind))

    if not passing:
        print("\nNo hybrid arm passes every gate. That is the result: report the "
              "STS-vs-substitution trade-off curve, do not relax a threshold.")
        return None
    weight, kind = sorted(passing)[0]
    print(f"\nSELECTED_HYBRID = {kind}:{weight}   (smallest passing weight of "
          f"{len(passing)}; set it in the environment and rerun with RUN_MULTISEED)")
    return f"{kind}:{weight}"

GATE_CHOICE = hybrid_gate()


## Reporting

Main table: NTP, augmented NTP, Zhang set-marginal, alternative-only uniform, contrastive (if promoted). Control table: pretrained, randomized $\lambda=.25$, randomized $\lambda=1$ (matched weight), target-inclusive uniform ablation. Paired bootstrap intervals on every SWORDS comparison; mean ± sd over three seeds everywhere else. Do not expand to 3B from this notebook until the three-seed 1B result is in; the hierarchy experiment stays deferred.


## SWORDS test — one locked invocation

Everything above is SWORDS **dev**: $\alpha$, $\beta$ and $\gamma$ were all chosen on it, so it is development data and cannot support the headline number. This cell scores test **once**, over every locked arm in a single call, so the method and its baselines are measured on identical rows with identical code.

It refuses to run until `SELECTED_HYBRID` is set, and it writes `swords_test_locked.json` recording the arms and the commit. If that file already exists the cell stops: a second test pass with a changed method is the one thing this protocol exists to prevent. Nothing here may be re-run after reading the result.


In [ ]:
RUN_SWORDS_TEST = False   # set True exactly once, after the method is locked
if RUN_SWORDS_TEST:
    assert SELECTED_HYBRID, "lock the method first: the gate sets SELECTED_HYBRID"
    marker = SCREEN_DIR / "swords_test_locked.json"
    assert not marker.is_file(), (
        f"SWORDS test has already been run: {marker}. Re-running after seeing the "
        "result invalidates it. Delete the marker ONLY if the previous run crashed.")

    # One invocation, every locked arm, in a fixed order.  Baselines come from
    # Task 15's manifest so the test table cannot quietly use a different NTP
    # than the dev table did.
    task15_runs = restore_all(load_runs(f"task15{_TAG_SUFFIX}"))
    locked = {"pretrained": BASE_MODEL}
    for family in ("ntp", "augmented_ntp", "zhang", "randomized"):
        for seed in (42, 123, 2024):
            key = f"{family}_seed{seed}"
            if key in task15_runs:
                locked[key] = str(task15_runs[key])
            else:
                print("MISSING baseline, test table will be incomplete:", key)
    OBJECTIVE_RUNS = restore_all(OBJECTIVE_RUNS)
    for seed in (42, 123, 2024):
        key = f"calibrated_seed{seed}"
        if key in OBJECTIVE_RUNS:
            locked[key] = str(OBJECTIVE_RUNS[key])
        else:
            print("MISSING method seed:", key)
    # The two ablations the paper reports beside the method.
    for key in ("alternative_uniform_seed42", "inclusive_uniform_alpha0.5_seed42"):
        if key in OBJECTIVE_RUNS:
            locked[key] = str(OBJECTIVE_RUNS[key])

    checkpoints = list(locked.values())
    print(f"scoring {len(checkpoints)} locked checkpoints on SWORDS TEST")
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
         "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL,
         "--base_model", BASE_MODEL,
         "--swords_json", MAIN / "data/swords/swords-v1.1_test.json.gz",
         "--results_json", SCREEN_DIR / "swords_test.json",
         "--modes", "left", "full"], cwd=MAIN)

    # Seed-matched paired intervals: each method seed against the SAME seed of
    # each baseline.  Averaging over mismatched seeds would fold seed variance
    # into the effect.
    for family in ("zhang", "ntp", "augmented_ntp"):
        for seed in (42, 123, 2024):
            key = f"{family}_seed{seed}"
            if key not in locked:
                continue
            run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
                 "--kind", "swords", "--results-json", SCREEN_DIR / "swords_test.json",
                 "--baseline-index", checkpoints.index(locked[key]),
                 "--output", SCREEN_DIR / f"swords_test_paired_ci_vs_{key}.json"], cwd=MAIN)

    marker.write_text(json.dumps({
        "selected_hybrid": SELECTED_HYBRID,
        "arms": locked,
        "upstream_commit": UPSTREAM_COMMIT,
        "written": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }, indent=2))
    print("locked:", marker)


### Reporting the test table

For each metric report (a) mean ± sd over the three seeds of each arm, (b) the seed-matched difference method$_s$ − baseline$_s$ for $s \in \{42,123,2024\}$, and (c) a paired bootstrap over per-target scores, averaged across seeds — not a bootstrap over the seed means, which has three points and no power. A claim needs all three seeds to agree in sign with intervals excluding zero.
